In [1]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [2]:
input_size = 1
hidden_size = 32
num_layers = 1
num_classes = 2

sequence_length = 10

batch_size = 16
learning_rate = 0.001
epochs = 20

In [3]:
torch.manual_seed(42)

X = torch.rand(1000, sequence_length, input_size)

y = (X.sum(dim=1) > 5).long().squeeze()

In [4]:
train_size = 800

X_train = X[:train_size]
y_train = y[:train_size]

X_test = X[train_size:]
y_test = y[train_size:]

In [5]:
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [6]:
class RNNModel(nn.Module):

    def __init__(self):
        super(RNNModel, self).__init__()

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):

        h0 = torch.zeros(
            num_layers,
            x.size(0),
            hidden_size
        )

        out, hidden = self.rnn(x, h0)

        out = out[:, -1, :]

        out = self.fc(out)

        return out

In [7]:
model = RNNModel()

print(model)

RNNModel(
  (rnn): RNN(1, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=2, bias=True)
)


In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate
)

In [9]:
for epoch in range(epochs):

    model.train()

    running_loss = 0

    for sequences, labels in train_loader:

        outputs = model(sequences)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs}, "
        f"Loss = {running_loss/len(train_loader):.4f}"
    )

Epoch 1/20, Loss = 0.6880
Epoch 2/20, Loss = 0.5974
Epoch 3/20, Loss = 0.4321
Epoch 4/20, Loss = 0.3560
Epoch 5/20, Loss = 0.2957
Epoch 6/20, Loss = 0.3148
Epoch 7/20, Loss = 0.2465
Epoch 8/20, Loss = 0.2420
Epoch 9/20, Loss = 0.2238
Epoch 10/20, Loss = 0.2271
Epoch 11/20, Loss = 0.2051
Epoch 12/20, Loss = 0.2080
Epoch 13/20, Loss = 0.1821
Epoch 14/20, Loss = 0.1794
Epoch 15/20, Loss = 0.1797
Epoch 16/20, Loss = 0.1645
Epoch 17/20, Loss = 0.1648
Epoch 18/20, Loss = 0.1580
Epoch 19/20, Loss = 0.1431
Epoch 20/20, Loss = 0.1634


In [10]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for sequences, labels in test_loader:

        outputs = model(sequences)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 97.00%


In [11]:
sample = X_test[0].unsqueeze(0)

model.eval()

with torch.no_grad():

    output = model(sample)

    prediction = torch.argmax(output, dim=1)

print("Prediction :", prediction.item())
print("Actual Label:", y_test[0].item())

Prediction : 0
Actual Label: 0
